# Sigchos Agrotech — Entrenamiento del modelo TFLite

Entrena un clasificador de enfermedades foliares de zapallo (MobileNetV2 + transfer learning) usando el dataset de Kaggle `tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh`, y lo exporta a `.tflite` listo para `mobile_app/assets/ml/`.

**Antes de correr:** Runtime → Change runtime type → GPU (T4).

**Necesitas:** tu API token de Kaggle. Consíguelo en https://www.kaggle.com/settings → API → "Create New Token" (descarga `kaggle.json`).

## 1. Instalar dependencias

In [ ]:
!pip install -q kagglehub tensorflow

## 2. Autenticar y descargar el dataset

Al ejecutar la celda te va a pedir usuario y key de Kaggle (los de tu `kaggle.json`).

In [ ]:
import kagglehub

kagglehub.login()
dataset_path = kagglehub.dataset_download("tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh")
print("Dataset en:", dataset_path)

## 3. Inspeccionar carpetas del dataset

El dataset trae 5 clases (no 6): sana, mildiu velloso, oídio, mosaico y mancha bacteriana — no incluye "daño por plaga". Ejecuta esta celda y revisa los nombres reales de carpeta antes de continuar; ajusta `MAPEO_CARPETAS` en la celda 4 si no calzan.

In [ ]:
import os

for root, dirs, files in os.walk(dataset_path):
    nivel = root.replace(dataset_path, '').count(os.sep)
    if nivel <= 2:
        print('  ' * nivel + os.path.basename(root) + f'  ({len(files)} archivos)')

## 4. Mapear carpetas del dataset → clases de la app

**Ajusta las claves (lado izquierdo) para que coincidan EXACTO con los nombres de carpeta impresos arriba.** Los valores (lado derecho) son los `claseId` que ya usa la app (`mobile_app/lib/core/constants/enfermedades.dart`) — no los cambies.

In [ ]:
MAPEO_CARPETAS = {
    "Healthy Leaf": "hoja_sana",
    "Downy Mildew": "mildiu",
    "Powdery Mildew": "oidio",
    "Mosaic Disease": "amarillamiento",
    "Bacterial Leaf Spot": "mancha_foliar",
}
# "dano_plaga" queda fuera de este modelo v1: el dataset no trae esa clase.
# La app la sigue mostrando en Recomendaciones/Historial, pero el clasificador
# nunca la va a predecir hasta que se sume un dataset con esa clase.

## 5. Reorganizar imágenes en carpetas por claseId

In [ ]:
import shutil, glob

DATASET_LISTO = "/content/dataset_listo"
shutil.rmtree(DATASET_LISTO, ignore_errors=True)
os.makedirs(DATASET_LISTO, exist_ok=True)

def _normalizar(s):
    return s.strip().lower().replace('_', ' ').replace('-', ' ')

EXTS = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
carpetas_disponibles = {}
for root, dirs, files_ in os.walk(dataset_path):
    imgs = [f for f in files_ if f.endswith(EXTS)]
    if imgs:
        carpetas_disponibles[_normalizar(os.path.basename(root))] = (root, len(imgs))

print("Carpetas con imágenes encontradas en el dataset:")
for nombre_normalizado, (ruta, n) in carpetas_disponibles.items():
    print(f"  '{nombre_normalizado}' -> {ruta}  ({n} imágenes)")

for carpeta_original, clase_id in MAPEO_CARPETAS.items():
    clave = _normalizar(carpeta_original)
    if clave not in carpetas_disponibles:
        raise ValueError(
            f"No se encontró la carpeta '{carpeta_original}' (normalizada: '{clave}'). "
            f"Copia el nombre EXACTO de la lista impresa arriba y ajusta MAPEO_CARPETAS (celda 4)."
        )
    origen, _ = carpetas_disponibles[clave]
    destino = os.path.join(DATASET_LISTO, clase_id)
    shutil.copytree(origen, destino, dirs_exist_ok=True)
    print(f"{clase_id}: {len(os.listdir(destino))} imágenes")

## 6. Cargar datasets de entrenamiento/validación

In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_LISTO, validation_split=0.2, subset="training", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_LISTO, validation_split=0.2, subset="validation", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
)

CLASES = train_ds.class_names  # orden alfabético — define el orden de salida del modelo
print("Clases (orden del modelo):", CLASES)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 7. Construir el modelo (MobileNetV2 + transfer learning)

Usa `preprocess_input` de MobileNetV2 — es exactamente la normalización `(pixel-127.5)/127.5` que ya implementa `mobile_app/lib/services/tflite_service.dart`, así que no hace falta tocar el código Flutter.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.1),
])

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(CLASES), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 8. Entrenar (fase 1: solo la cabeza nueva)

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=12)

## 9. Fine-tuning (fase 2: descongelar las últimas capas)

In [ ]:
base_model.trainable = True
for capa in base_model.layers[:-30]:
    capa.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_fine = model.fit(train_ds, validation_data=val_ds, epochs=8)

## 10. Evaluar (matriz de confusión — útil para la sección "Modelo IA" del panel admin)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for imagenes, etiquetas in val_ds:
    preds = model.predict(imagenes, verbose=0)
    y_true.extend(etiquetas.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=CLASES))
print("Matriz de confusión:\n", confusion_matrix(y_true, y_pred))

## 11. Convertir a TensorFlow Lite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("model.tflite", "wb") as f:
    f.write(tflite_model)

with open("labels.txt", "w") as f:
    f.write("\n".join(CLASES))

print("model.tflite:", len(tflite_model) / 1024, "KB")
print("labels.txt (orden del modelo):", CLASES)

## 12. Descargar y reemplazar en el proyecto Flutter

In [ ]:
from google.colab import files

files.download("model.tflite")
files.download("labels.txt")

**Último paso (en tu PC, no en Colab):** copia los dos archivos descargados a `mobile_app/assets/ml/`, reemplazando los placeholders. Luego:
```
cd mobile_app
flutter clean
flutter pub get
flutter run
```
`TFLiteService` (`mobile_app/lib/services/tflite_service.dart`) detecta el modelo real automáticamente y deja de usar el modo simulado — no requiere ningún cambio de código.